In [0]:
data = [
        (1,"John","USA","IT",25000,32),
        (2,"Mary","USA","HR",27000,28),
        (3,"Jose","USA","IT",30000,35),
        (4,"Karl","USA","IT",32000,40),
        (5,"Sara","USA","IT",35000,45),
        (6,"Sergio","USA","IT",38000,50),
        (7,"Pedro","USA","IT",40000,55)
]
df = spark.createDataFrame(data,["emp_id","name","country","dep","salary","age"])

### Check the schema before upsert

In [0]:
# Set the session catalog context using SQL
spark.sql("USE CATALOG `databricks-pyspark`")

In [0]:
spark.sql("USE SCHEMA `databricks-pyspark-schema`")

In [0]:
%sql 
SHOW CATALOGS;

In [0]:
df.write.mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("employee_source")

In [0]:
%sql
select * from employee_source;

In [0]:
target_df = spark.table("employee_source")

In [0]:
target_df.write \
       .mode("overwrite") \
           .format("delta") \
               .saveAsTable("employee_target")


In [0]:
%sql
select * from employee_target;

In [0]:
data = [
        (1,"John","USA","IT",32000,32),
        (2,"Mary","USA","HR",38000,28),
        (10,"leanne","USA","IT",35000,45),
        (11,"maria","USA","IT",38000,50),
        (12,"pedro","USA","IT",400000,45)
]
updated_df = spark.createDataFrame(data,["emp_id","name","country","dep","salary","age"])

In [0]:
updated_df.write \
    .mode("overwrite") \
        .format("delta") \
        .saveAsTable("employee_source")

In [0]:
%sql 
select * from employee_source;

### as this incremental load we can have changed data

In [0]:
spark.sql("""
          MERGE INTO employee_target t
          USING employee_source s
          ON t.emp_id = s.emp_id
          WHEN MATCHED THEN
          UPDATE set t.name = s.name,t.salary = s.salary,t.age = s.age,t.dep = s.dep,t.country = s.country
          WHEN NOT MATCHED THEN
          INSERT (emp_id,name,country,dep,salary,age) VALUES (s.emp_id,s.name,s.country,s.dep,s.salary,s.age)
          
          """)

In [0]:
%sql
select * from employee_target t;